# Figure S25 — Amphibole barometry

## 1. Imports and shared 00_Setup style

This notebook uses the current S1 supplemental workbook, Arial typography, 8-point ordinary text, 9-point bold uppercase panel labels, full plot boxes, light gridlines, and editable PDF/SVG text. The code is linear and contains no custom functions.

In [1]:
from pathlib import Path
import hashlib
import inspect
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from packaging.version import Version
from PIL import Image
import Thermobar as pt

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = MAXIMUM_FIGURE_HEIGHT_IN
BASE_TEXT_SIZE_PT = 8.0
PANEL_LABEL_SIZE_PT = 9.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [PUBLICATION_FONT],
    "font.size": BASE_TEXT_SIZE_PT,
    "axes.labelsize": BASE_TEXT_SIZE_PT,
    "axes.titlesize": BASE_TEXT_SIZE_PT,
    "xtick.labelsize": BASE_TEXT_SIZE_PT,
    "ytick.labelsize": BASE_TEXT_SIZE_PT,
    "legend.fontsize": 6.0,
    "legend.title_fontsize": BASE_TEXT_SIZE_PT,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 0.5,
    "lines.linewidth": 0.75,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "savefig.dpi": EXPORT_DPI,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


## 2. Figure paths and shared sample configuration

S24 uses the current S1 workbook only. No legacy or external data files are read.

In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_S25_Amph_Barometry"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_S25_Amph_Barometry_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = WORK_ROOT / "DataSheets" / "Updated_Oct20" / "Dechert_S1_AllData.xlsx"
if not S1_PATH.is_file():
    raise FileNotFoundError(f"Missing S1 workbook: {S1_PATH}")
SOURCE_HASH_BEFORE = hashlib.sha256(S1_PATH.read_bytes()).hexdigest()

SAMPLE_COLORS = {"74D": "#F4A261", "74A": "#D55E00", "75": "#56B4E9", "76A": "#0072B2"}
SAMPLE_LABELS = {"74D": "B Middle", "74A": "B Base", "75": "G Top", "76A": "G Middle"}
GROUP_ORDER = [
    "High-Al2O3 transects", "Other transect points",
    "B Mingled Dark", "B Mingled Light", "G Mingled Dark", "G Mingled Light",
]
MINGLED_COLORS = {
    "B Mingled Dark": "#808080", "B Mingled Light": "#EE3377",
    "G Mingled Dark": "#000000", "G Mingled Light": "#66C2C2",
}


## 3. Read and prepare the established S1 populations

In [3]:
transect_source = pd.read_excel(S1_PATH, sheet_name="Transects")
mingled_source = pd.read_excel(S1_PATH, sheet_name="Mingled_Minerals")

all_linescans = transect_source.copy().rename(columns={"Sample": "Sample_Group"})
scan_counts = all_linescans.groupby("Linescan_ID")["Predict_Mineral"].value_counts(dropna=False)
scan_total = all_linescans.groupby("Linescan_ID").size()
dominant = scan_counts.groupby(level=0).idxmax().map(lambda value: value[1])
dominant_fraction = scan_counts.groupby(level=0).max().div(scan_total)
all_linescans["Dominant_Mineral"] = all_linescans["Linescan_ID"].map(dominant)
all_linescans["Dominant_Mineral_Fraction"] = all_linescans["Linescan_ID"].map(dominant_fraction)
assert all_linescans["Sample_Group"].isin(["74A", "74D", "75", "76A"]).all()

RAW_OXIDE_MAP = {
    "SiO2": "SiO2_SEMMineral_Raw", "TiO2": "TiO2_SEMMineral_Raw",
    "Al2O3": "Al2O3_SEMMineral_Raw", "FeOt": "FeOt_SEMMineral_Raw",
    "MnO": "MnO_SEMMineral_Raw", "MgO": "MgO_SEMMineral_Raw",
    "CaO": "CaO_SEMMineral_Raw", "Na2O": "Na2O_SEMMineral_Raw",
    "K2O": "K2O_SEMMineral_Raw",
}
MINGLED_RAW_OXIDE_MAP = {
    "SiO2": "SiO2_SEMMineral_Raw", "TiO2": "TiO2_SEMMineral_Raw",
    "Al2O3": "Al2O3_SEMMineral_Raw", "FeOt": "FeOt_SEMMineral_Raw",
    "MnO": "MnO_SEMMineral_Raw", "MgO": "MgO_SEMMineral_Raw",
    "CaO": "CaO_SEMMineral_Raw", "Na2O": "Na2O_SEMMineral_Raw",
    "K2O": "K2O_SEMMineral_Raw", "Cr2O3": "Cr2O3_SEMMineral_Raw",
}

linescan_score = pd.to_numeric(all_linescans["Prediction_Score"], errors="coerce")
transect_mask = (
    all_linescans["Dominant_Mineral"].astype("string").str.casefold().eq("amphibole")
    & all_linescans["Dominant_Mineral_Fraction"].ge(0.95)
    & all_linescans["Predict_Mineral"].astype("string").str.casefold().eq("amphibole")
    & linescan_score.gt(0.85)
    & pd.to_numeric(all_linescans["Distance (µm)"], errors="coerce").notna()
)
transects = all_linescans.loc[transect_mask].copy()
transects["Source Data Row"] = transects.index
for oxide, source_column in RAW_OXIDE_MAP.items():
    transects[f"{oxide}_Thermobar"] = pd.to_numeric(transects[source_column], errors="coerce")
transects["Cr2O3_Thermobar"] = 0.0
transects["Cl_Thermobar"] = pd.to_numeric(transects["Cl_SEMMineral_Raw"], errors="coerce").fillna(0.0)
transects = transects.dropna(subset=[f"{oxide}_Thermobar" for oxide in RAW_OXIDE_MAP]).reset_index(drop=True)
transects["Analysis ID"] = [
    f"transect__{line_id}__datasheetrow_{int(row)}"
    for line_id, row in zip(transects["Linescan_ID"], transects["Source Data Row"])
]
HIGH_AL_LINES = {
    "74A2_amph12_linescan_lr",
    "74A2_amph7_linescan_rl",
    "74A2_px14_linescan_lr",
}
transects["Group"] = np.where(
    transects["Linescan_ID"].isin(HIGH_AL_LINES),
    GROUP_ORDER[0],
    GROUP_ORDER[1],
)
transects["Position"] = "Transect"
transects["Dataset type"] = "Transect"
transects["Color"] = transects["Sample_Group"].map(SAMPLE_COLORS)
transects.loc[transects["Group"].eq(GROUP_ORDER[0]), "Color"] = "#39FF14"

mingled_minerals = mingled_source.copy().rename(columns={"Type of Pumice": "Mafic/Felsic"})
mingled_score = pd.to_numeric(mingled_minerals["Prediction_Score"], errors="coerce")
mingled_mask = (
    mingled_minerals["Predict_Mineral"].astype("string").str.casefold().eq("amphibole")
    & mingled_score.gt(0.85)
    & mingled_minerals["Dataset"].isin(["74Ming", "76Ming"])
)
mingled = mingled_minerals.loc[mingled_mask].copy()
mingled["Source Data Row"] = mingled.index
for oxide, source_column in MINGLED_RAW_OXIDE_MAP.items():
    mingled[f"{oxide}_Thermobar"] = pd.to_numeric(mingled[source_column], errors="coerce")
mingled = mingled.dropna(subset=[f"{oxide}_Thermobar" for oxide in MINGLED_RAW_OXIDE_MAP]).reset_index(drop=True)
mingled[list(MINGLED_RAW_OXIDE_MAP.values())] = mingled[list(MINGLED_RAW_OXIDE_MAP.values())].clip(lower=0)
texture = mingled["Mafic/Felsic"].astype("string").str.strip().str.casefold()
mingled["Group"] = np.select(
    [
        mingled["Dataset"].eq("74Ming") & (texture.str.startswith("m") | texture.eq("dark")),
        mingled["Dataset"].eq("74Ming") & (texture.str.startswith("f") | texture.eq("light")),
        mingled["Dataset"].eq("76Ming") & (texture.str.startswith("m") | texture.eq("dark")),
        mingled["Dataset"].eq("76Ming") & (texture.str.startswith("f") | texture.eq("light")),
    ],
    GROUP_ORDER[2:],
    default="Unassigned",
)
if mingled["Group"].eq("Unassigned").any():
    raise ValueError("A retained mingled amphibole could not be assigned to a Dark/Light group.")
mingled["Position"] = mingled["Crystal Position"].fillna("Other").replace("", "Other")
mingled.loc[~mingled["Position"].isin(["Core", "Rim", "Other"]), "Position"] = "Other"
mingled["Analysis ID"] = [
    f"mingled__{dataset}__{name}__datasheetrow_{int(row)}"
    for dataset, name, row in zip(mingled["Dataset"], mingled["Sample Name"], mingled["Source Data Row"])
]
mingled["Color"] = mingled["Group"].map(MINGLED_COLORS)
mingled["Dataset type"] = "Mingled"

assert set(transects.loc[transects["Group"].eq(GROUP_ORDER[0]), "Linescan_ID"]) == HIGH_AL_LINES
assert set(mingled["Group"]) == set(GROUP_ORDER[2:])


## 4. Recalculate the four source pressure outputs and selected Ridolfi P–T fields

The original Ridolfi (2021), Mutch (2016), Médard (2022; Ridolfi sites), and Ridolfi (2012) calls are retained. Filtered and diagnostic-inclusive Ridolfi results remain separate.

In [4]:
if Version(pd.__version__) >= Version("3.0"):
    import Thermobar.core as tb_core
    import Thermobar.amphibole as tb_amphibole
    for function_name in ("calculate_sites_ridolfi", "get_amp_sites_leake", "get_amp_sites_avferric_zhang"):
        original = getattr(tb_core, function_name)
        try:
            source = inspect.getsource(original)
        except OSError:
            continue
        float_source = re.sub(r"(norm_cations\[['\"][^'\"]+['\"]\]\s*)=0(?![.\d])", r"\1=0.0", source)
        if float_source == source:
            raise RuntimeError(f"Thermobar compatibility patch found no integer initializer in {function_name}.")
        exec(float_source, tb_core.__dict__)
    tb_amphibole.calculate_sites_ridolfi = tb_core.calculate_sites_ridolfi
    tb_amphibole.get_amp_sites_avferric_zhang = tb_core.get_amp_sites_avferric_zhang

transect_comps = pd.DataFrame({f"{oxide}_Amp": transects[f"{oxide}_Thermobar"].astype(float) for oxide in RAW_OXIDE_MAP})
transect_comps["Cr2O3_Amp"] = transects["Cr2O3_Thermobar"].astype(float)
transect_comps["F_Amp"] = 0.0
transect_comps["Cl_Amp"] = transects["Cl_Thermobar"].astype(float)
transect_comps["Sample_ID_Amp"] = transects["Analysis ID"].values

mingled_comps = pd.DataFrame({f"{oxide}_Amp": mingled[f"{oxide}_Thermobar"].astype(float) for oxide in MINGLED_RAW_OXIDE_MAP})
mingled_comps["F_Amp"] = 0.0
mingled_comps["Cl_Amp"] = 0.0
mingled_comps["Sample_ID_Amp"] = mingled["Analysis ID"].values

transect_PT = pt.calculate_amp_only_press_temp(amp_comps=transect_comps, equationP="P_Ridolfi2021", equationT="T_Ridolfi2012", Ridolfi_Filter=False).reset_index(drop=True)
transect_Ridolfi_filtered = pt.calculate_amp_only_press(amp_comps=transect_comps, equationP="P_Ridolfi2021", classification=True, Ridolfi_Filter=True).reset_index(drop=True)
transect_Ridolfi_diagnostic = pt.calculate_amp_only_press(amp_comps=transect_comps, equationP="P_Ridolfi2021", classification=True, Ridolfi_Filter=False).reset_index(drop=True)
transect_Mutch = pt.calculate_amp_only_press(amp_comps=transect_comps, equationP="P_Mutch2016").reset_index(drop=True)
transect_Medard = pt.calculate_amp_only_press(amp_comps=transect_comps, equationP="P_Medard2022_RidolfiSites").reset_index(drop=True)

mingled_PT = pt.calculate_amp_only_press_temp(amp_comps=mingled_comps, equationP="P_Ridolfi2021", equationT="T_Ridolfi2012", Ridolfi_Filter=False).reset_index(drop=True)
mingled_Ridolfi_filtered = pt.calculate_amp_only_press(amp_comps=mingled_comps, equationP="P_Ridolfi2021", classification=True, Ridolfi_Filter=True).reset_index(drop=True)
mingled_Ridolfi_diagnostic = pt.calculate_amp_only_press(amp_comps=mingled_comps, equationP="P_Ridolfi2021", classification=True, Ridolfi_Filter=False).reset_index(drop=True)
mingled_Mutch = pt.calculate_amp_only_press(amp_comps=mingled_comps, equationP="P_Mutch2016").reset_index(drop=True)
mingled_Medard = pt.calculate_amp_only_press(amp_comps=mingled_comps, equationP="P_Medard2022_RidolfiSites").reset_index(drop=True)

transect_pressures = pd.DataFrame({
    "T_C_Ridolfi2012": pd.to_numeric(transect_PT["T_K_calc"], errors="coerce") - 273.15,
    "Ridolfi_Input_Check": transect_Ridolfi_diagnostic["Input_Check"].fillna(False).astype(bool),
    "Ridolfi_equation_selected": transect_Ridolfi_diagnostic["equation"].astype("string"),
    "P_kbar_Ridolfi2021_filtered": pd.to_numeric(transect_Ridolfi_filtered["P_kbar_calc"], errors="coerce"),
    "P_kbar_Ridolfi2021_diagnostic": pd.to_numeric(transect_Ridolfi_diagnostic["P_kbar_calc"], errors="coerce"),
    "P_kbar_Mutch2016": pd.to_numeric(transect_Mutch["P_kbar_calc"], errors="coerce"),
    "P_kbar_Medard2022_RidolfiSites": pd.to_numeric(transect_Medard, errors="coerce"),
})
mingled_pressures = pd.DataFrame({
    "T_C_Ridolfi2012": pd.to_numeric(mingled_PT["T_K_calc"], errors="coerce") - 273.15,
    "Ridolfi_Input_Check": mingled_Ridolfi_diagnostic["Input_Check"].fillna(False).astype(bool),
    "Ridolfi_equation_selected": mingled_Ridolfi_diagnostic["equation"].astype("string"),
    "P_kbar_Ridolfi2021_filtered": pd.to_numeric(mingled_Ridolfi_filtered["P_kbar_calc"], errors="coerce"),
    "P_kbar_Ridolfi2021_diagnostic": pd.to_numeric(mingled_Ridolfi_diagnostic["P_kbar_calc"], errors="coerce"),
    "P_kbar_Mutch2016": pd.to_numeric(mingled_Mutch["P_kbar_calc"], errors="coerce"),
    "P_kbar_Medard2022_RidolfiSites": pd.to_numeric(mingled_Medard, errors="coerce"),
})

transect_wide = transects[["Analysis ID", "Group", "Dataset type", "Position", "Color"]].copy().reset_index(drop=True)
mingled_wide = mingled[["Analysis ID", "Group", "Dataset type", "Position", "Color"]].copy().reset_index(drop=True)
for column in transect_pressures.columns:
    transect_wide[column] = transect_pressures[column].to_numpy()
for column in mingled_pressures.columns:
    mingled_wide[column] = mingled_pressures[column].to_numpy()
all_wide = pd.concat([transect_wide, mingled_wide], ignore_index=True)
all_wide["Group"] = pd.Categorical(all_wide["Group"], GROUP_ORDER, ordered=True)
all_wide["Jitter"] = np.random.default_rng(20260913).uniform(-0.22, 0.22, len(all_wide))

PRESSURE_MODELS = {
    "Ridolfi 2021 (filtered)": "P_kbar_Ridolfi2021_filtered",
    "Ridolfi 2021 (all; diagnostic)": "P_kbar_Ridolfi2021_diagnostic",
    "Mutch et al. (2016)": "P_kbar_Mutch2016",
    "Médard et al. (2022; Ridolfi sites)": "P_kbar_Medard2022_RidolfiSites",
}
pressure_long_parts = []
for model, column in PRESSURE_MODELS.items():
    part = all_wide.copy()
    part["Pressure model"] = model
    part["Pressure (kbar)"] = pd.to_numeric(part[column], errors="coerce")
    part["Finite pressure"] = np.isfinite(part["Pressure (kbar)"])
    part["Passes Ridolfi input screen"] = part["Finite pressure"] & part["Ridolfi_Input_Check"]
    pressure_long_parts.append(part)
pressure_long = pd.concat(pressure_long_parts, ignore_index=True)
pressure_long["Pressure model"] = pd.Categorical(pressure_long["Pressure model"], list(PRESSURE_MODELS), ordered=True)

RIDOLFI_EQUATION_COLORS = {
    equation: color for equation, color in zip(
        sorted(all_wide["Ridolfi_equation_selected"].dropna().astype(str).unique()),
        ["#5E3C99", "#B2182B", "#C51B7D", "#7A0177", "#EF3B2C"],
    )
}
assert set(transects.loc[transects["Group"].eq(GROUP_ORDER[0]), "Linescan_ID"]) == HIGH_AL_LINES
assert set(mingled["Group"]) == set(GROUP_ORDER[2:])


## 5. Plot and export Figure S25

In [5]:
AMPHIBOLE_MARKER = "^"
GROUP_POSITIONS = {group: index for index, group in enumerate(GROUP_ORDER)}
XTICK_LABELS = [
    "High-Al$_2$O$_3$\ntransects", "Other transect\npoints",
    "B Mingled\nDark", "B Mingled\nLight", "G Mingled\nDark", "G Mingled\nLight",
]

figure = plt.figure(figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN))
grid = figure.add_gridspec(4, 2, height_ratios=[1.0, 1.0, 0.22, 1.35], hspace=0.30, wspace=0.25)
pressure_axes = [figure.add_subplot(grid[0, 0]), figure.add_subplot(grid[0, 1]), figure.add_subplot(grid[1, 0]), figure.add_subplot(grid[1, 1])]
group_legend_axis = figure.add_subplot(grid[2, :])
group_legend_axis.axis("off")
pt_axis = figure.add_subplot(grid[3, :])

finite_pressure_values = pressure_long.loc[pressure_long["Finite pressure"], "Pressure (kbar)"]
pressure_padding = max(0.15, float(finite_pressure_values.max() - finite_pressure_values.min()) * 0.04)
pressure_ymin = 0.5 * np.floor((finite_pressure_values.min() - pressure_padding) / 0.5)
pressure_ymax = 0.5 * np.ceil((finite_pressure_values.max() + pressure_padding) / 0.5)

for panel_index, (axis, model) in enumerate(zip(pressure_axes, PRESSURE_MODELS)):
    model_data = pressure_long.loc[pressure_long["Pressure model"].eq(model)]
    for group in GROUP_ORDER:
        group_data = model_data.loc[model_data["Group"].eq(group)]
        for passed, selected in group_data.groupby("Ridolfi_Input_Check", sort=False):
            selected = selected.loc[selected["Finite pressure"]]
            if selected.empty:
                continue
            axis.scatter(
                GROUP_POSITIONS[group] + selected["Jitter"],
                selected["Pressure (kbar)"],
                s=28.0, marker=AMPHIBOLE_MARKER,
                facecolors=selected["Color"] if passed else "none",
                edgecolors="black" if passed else selected["Color"],
                linewidths=MINIMUM_LINE_WIDTH_PT if passed else 0.52,
                alpha=1.0, zorder=2,
            )
        median_values = group_data.loc[group_data["Passes Ridolfi input screen"], "Pressure (kbar)"]
        if median_values.notna().any():
            axis.hlines(
                median_values.median(), GROUP_POSITIONS[group] - 0.29,
                GROUP_POSITIONS[group] + 0.29, color="black", linewidth=1.50, zorder=6,
            )
    axis.set_xlim(-0.5, len(GROUP_ORDER) - 0.5)
    axis.set_ylim(pressure_ymin, pressure_ymax)
    axis.set_ylabel("Pressure (kbar)")
    axis.set_title(model, pad=2.2)
    axis.text(-0.095, 1.025, f"{chr(65 + panel_index)})", transform=axis.transAxes,
              ha="left", va="bottom", fontsize=PANEL_LABEL_SIZE_PT,
              fontweight="bold", clip_on=False)
    axis.grid(True, axis="y", linestyle=":", linewidth=0.45, color="0.78", alpha=0.65)
    axis.set_axisbelow(True)
    axis.set_xticks(range(len(GROUP_ORDER)))
    if panel_index < 2:
        axis.set_xticklabels([])
    else:
        axis.set_xticklabels(XTICK_LABELS, rotation=25, ha="right", fontsize=6)
        axis.tick_params(axis="x", pad=3.0)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.55)

pt_data = all_wide.copy()
pt_data["Temperature (°C)"] = pd.to_numeric(pt_data["T_C_Ridolfi2012"], errors="coerce")
pt_data["Selected pressure (kbar)"] = pd.to_numeric(pt_data["P_kbar_Ridolfi2021_diagnostic"], errors="coerce")
pt_data["Equation"] = pt_data["Ridolfi_equation_selected"].astype(str)
pt_data["Finite P-T"] = np.isfinite(pt_data["Temperature (°C)"]) & np.isfinite(pt_data["Selected pressure (kbar)"])
for equation, color in RIDOLFI_EQUATION_COLORS.items():
    equation_data = pt_data.loc[pt_data["Equation"].eq(equation)]
    for passed, selected in equation_data.groupby("Ridolfi_Input_Check", sort=False):
        selected = selected.loc[selected["Finite P-T"]]
        if selected.empty:
            continue
        pt_axis.scatter(
            selected["Temperature (°C)"], selected["Selected pressure (kbar)"],
            s=28.0, marker=AMPHIBOLE_MARKER,
            facecolors=color if passed else "none",
            edgecolors="black" if passed else color,
            linewidths=MINIMUM_LINE_WIDTH_PT if passed else 0.55,
            alpha=1.0, zorder=2,
        )

pt_axis.set_xlabel("Temperature (°C)")
pt_axis.set_ylabel("Pressure (kbar)")
pt_axis.set_title("Ridolfi 2012 temperature and selected Ridolfi 2021 pressure", pad=2.5)
pt_axis.text(-0.045, 1.025, "E)", transform=pt_axis.transAxes,
             ha="left", va="bottom", fontsize=PANEL_LABEL_SIZE_PT,
             fontweight="bold", clip_on=False)
pt_axis.grid(True, linestyle=":", linewidth=0.45, color="0.78", alpha=0.65)
pt_axis.set_axisbelow(True)
for spine in pt_axis.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(0.55)

group_legend_handles = [
    Patch(facecolor="#39FF14", edgecolor="black", linewidth=0.35, label="High-Al$_2$O$_3$ transects"),
    Patch(facecolor=SAMPLE_COLORS["74D"], edgecolor="black", linewidth=0.35, label="B Middle"),
    Patch(facecolor=SAMPLE_COLORS["74A"], edgecolor="black", linewidth=0.35, label="B Base"),
    Patch(facecolor=SAMPLE_COLORS["75"], edgecolor="black", linewidth=0.35, label="G Top"),
    Patch(facecolor=SAMPLE_COLORS["76A"], edgecolor="black", linewidth=0.35, label="G Middle"),
    Patch(facecolor=MINGLED_COLORS["B Mingled Dark"], edgecolor="black", linewidth=0.35, label="B Mingled Dark"),
    Patch(facecolor=MINGLED_COLORS["B Mingled Light"], edgecolor="black", linewidth=0.35, label="B Mingled Light"),
    Patch(facecolor=MINGLED_COLORS["G Mingled Dark"], edgecolor="black", linewidth=0.35, label="G Mingled Dark"),
    Patch(facecolor=MINGLED_COLORS["G Mingled Light"], edgecolor="black", linewidth=0.35, label="G Mingled Light"),
]
equation_legend_handles = [
    Patch(facecolor=color, edgecolor="black", linewidth=0.35, label=f"Ridolfi equation {equation}")
    for equation, color in RIDOLFI_EQUATION_COLORS.items()
]
status_handles = [
    Line2D([], [], marker=AMPHIBOLE_MARKER, linestyle="none", markersize=5.2, markerfacecolor="0.45", markeredgecolor="black", label="Amphibole: passes input screen"),
    Line2D([], [], marker=AMPHIBOLE_MARKER, linestyle="none", markersize=5.2, markerfacecolor="none", markeredgecolor="0.35", label="Amphibole: diagnostic only"),
    Line2D([], [], color="black", linewidth=1.50, label="Passing median (A–D)"),
]
upper_legend_handles = [group_legend_handles[i] for i in [1, 2, 3, 4, 5, 6, 7, 8]] + [group_legend_handles[0], status_handles[2]]
group_legend_axis.legend(
    handles=upper_legend_handles,
    loc="center", bbox_to_anchor=(0.50, 0.50), ncol=5,
    frameon=True, fancybox=False, framealpha=1.0, edgecolor="black",
    columnspacing=0.85, handletextpad=0.4, labelspacing=0.4, borderpad=0.45,
)
bottom_handles = equation_legend_handles + status_handles[:2]
figure.legend(
    handles=bottom_handles, loc="lower center", bbox_to_anchor=(0.50, 0.025),
    ncol=3, frameon=True, fancybox=False, framealpha=1.0, edgecolor="black",
    columnspacing=1.0, handletextpad=0.4, labelspacing=0.4, borderpad=0.45,
)
figure.subplots_adjust(left=0.105, right=0.988, top=0.975, bottom=0.14)

output_stem = OUTPUT_DIRECTORY / "Figure_S25_Amph_Barometry"
figure.savefig(output_stem.with_suffix(".pdf"), format="pdf", dpi=EXPORT_DPI, facecolor="white", transparent=False)
figure.savefig(output_stem.with_suffix(".png"), format="png", dpi=EXPORT_DPI, facecolor="white", transparent=False)
figure.savefig(output_stem.with_suffix(".svg"), format="svg", dpi=EXPORT_DPI, facecolor="white", transparent=False)
plt.close(figure)


## 6. Final export checks

In [6]:
assert hashlib.sha256(S1_PATH.read_bytes()).hexdigest() == SOURCE_HASH_BEFORE
assert len(pressure_axes) == 4
assert len(RIDOLFI_EQUATION_COLORS) > 0
assert all(axis.get_title(loc="left") == "" for axis in pressure_axes)
for suffix in (".pdf", ".png", ".svg"):
    output_path = OUTPUT_DIRECTORY / f"Figure_S25_Amph_Barometry{suffix}"
    assert output_path.is_file() and output_path.stat().st_size > 0
with Image.open(OUTPUT_DIRECTORY / "Figure_S25_Amph_Barometry.png") as image:
    assert image.size == (round(FIGURE_WIDTH_IN * EXPORT_DPI), round(FIGURE_HEIGHT_IN * EXPORT_DPI))
svg_text = (OUTPUT_DIRECTORY / "Figure_S25_Amph_Barometry.svg").read_text(encoding="utf-8")
assert "<text" in svg_text and "<image" not in svg_text.casefold()
assert (OUTPUT_DIRECTORY / "Figure_S25_Amph_Barometry.pdf").read_bytes().count(b"/Subtype /Type3") == 0
print("Figure S25 exported: PDF, PNG, and SVG.")


Figure S25 exported: PDF, PNG, and SVG.
